# Cymbal Foods: generate the dataset

**Pipeline step 10.** This notebook builds the Cymbal Foods dataset that every AlloyDB workshop lab loads, checks it, cuts it into the slices each lab module uses, and stages it in Cloud Storage.

Cymbal Foods is a fictional specialty food company that began more than a century ago as a single spice counter. Today it runs about 40 stores, an online store, and wholesale supply to about 4,000 restaurants on contract pricing. None of its data is real: the generator writes three years of it (Jan 2024 through Dec 2026) from a seed, so every run produces exactly the same rows.

| | |
| :-- | :-- |
| **Runs in** | Colab Enterprise (any runtime with 8 GB of memory or more), or any Python 3.11+ machine |
| **Reads** | the generator and its reference data from the workshop's GitHub repo (`datagen/`) |
| **Writes** | the local runtime, then `gs://class-demo/alloydb-labs/cymbalfoods-alloydb-workshop/dataset/v1/` |
| **Calls** | no models, no databases. Text generation is a separate step (notebook 11) |
| **Takes** | about 15 to 20 minutes; most of it is the year of website clickstream |

**To run it:** **Run all**, then copy the box the last cell prints and paste it back.

Each section explains what it does and why. The code that makes the data lives in `datagen/cfgen/`, one module per part of the business, so you can read how any table came to be.

---
## 1 · Configuration

The run flags first. Everything else that shapes the data (volumes, prices, seasonality, the planted teaching moments) is in `datagen/cfgen/config.py`, and the reference lists are in `datagen/content/`.

In [ ]:
# ===== RUN FLAGS: the only things you normally change ========================================
REGENERATE = True      # False = keep the data already staged and only re-cut the slices (minutes, not a regeneration)
SLICES     = "all"     # "all", or a comma-separated list of names from datagen/slices.yaml
STAGE      = True      # copy the result to the course bucket
SCALE      = 1         # 1 = lab scale. 10 (demo) needs a high-memory runtime
RUN_GA4    = True      # the 2026 GA4 clickstream (~22M events, about 9 of the run's minutes)
# =============================================================================================
REPO_URL   = "https://github.com/haggman/cymbalfoods-alloydb-workshop.git"
BRANCH     = "main"
GCS_ROOT   = "gs://class-demo/alloydb-labs/cymbalfoods-alloydb-workshop"
DATASET    = f"{GCS_ROOT}/dataset/v1" if SCALE == 1 else f"{GCS_ROOT}/dataset/v1-scale{SCALE:g}".replace(".", "p")
NEVER_WRITE = "gs://class-demo/alloydb-labs/cymbalgoal/"    # another team's labs read this; this notebook never writes there

In [ ]:
import datetime as dt, html, json, os, subprocess, sys, time
from pathlib import Path

RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
WORK = Path("/content/cymbalfoods") if Path("/content").exists() else Path.home() / "cymbalfoods"
WORK.mkdir(parents=True, exist_ok=True)
REPO = WORK / "repo"
OUT = WORK / "out"
HB = {}                      # everything for the handback, by section

def hb(section, line):
    HB.setdefault(section, []).append(str(line))
    print(line)

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and r.returncode:
        raise RuntimeError(f"{cmd}\n{r.stderr[-2000:]}")
    return r
hb("Run", f"run {RUN_ID}  regenerate={REGENERATE}  slices={SLICES}  stage={STAGE}  scale={SCALE}  ga4={RUN_GA4}")

---
## 2 · Get the generator

A shallow clone of the workshop repo, then the five libraries the generator needs (numpy, pandas, pyarrow, DuckDB, PyYAML). The library versions are recorded, because the same seed gives byte-identical files only with the same versions.

In [ ]:
S = "2 · Generator"
if REPO.exists():
    sh(f"git -C {REPO} fetch -q --depth 1 origin {BRANCH} && git -C {REPO} reset -q --hard origin/{BRANCH}")
else:
    sh(f"git clone -q --depth 1 --branch {BRANCH} {REPO_URL} {REPO}")
DATAGEN = REPO / "datagen"
sh(f"{sys.executable} -m pip install -q -r {DATAGEN / 'requirements.txt'}")
commit = sh(f"git -C {REPO} rev-parse --short HEAD").stdout.strip()
import numpy, pandas, pyarrow, duckdb
hb(S, f"repo {REPO_URL} @ {commit}")
hb(S, f"python {sys.version.split()[0]}  numpy {numpy.__version__}  pandas {pandas.__version__}  pyarrow {pyarrow.__version__}  duckdb {duckdb.__version__}")
hb(S, f"runtime: {os.cpu_count()} vCPU, {round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30, 1)} GB memory")

---
## 3 · The bucket guard

Every Cloud Storage call goes through `gcs()`. Writes are allowed **only** under this dataset's prefix, and anything under `alloydb-labs/cymbalgoal/` is refused, because another team's labs read it. The two prefixes share a stem, so the check matches on a trailing `/`, never on a bare string prefix.

The probe writes one tiny object to prove this identity can write. If it fails, the Qwiklabs project's account needs write access to the workshop prefix (the command is printed).

In [ ]:
S = "3 · Bucket"
GCS_LOG = []
_WRITE_VERBS = {"cp", "rsync", "rm"}

def _under(uri, prefix):
    p = prefix.rstrip("/")
    return uri == p or uri.startswith(p + "/")

def gcs(*args, check=True):
    args = list(args)
    verb = args[0]
    uris = [a for a in args[1:] if a.startswith("gs://")]
    if any(_under(u, NEVER_WRITE) for u in uris) and verb in _WRITE_VERBS:
        raise PermissionError(f"gcs(): refusing {verb} touching {NEVER_WRITE}")
    if verb in _WRITE_VERBS:
        dest = uris[-1] if verb != "rm" else uris[0]
        if not _under(dest, DATASET):
            raise PermissionError(f"gcs(): {verb} destination {dest} is outside {DATASET}")
    GCS_LOG.append(" ".join(["gcloud storage"] + args))
    r = subprocess.run(["gcloud", "storage"] + args, capture_output=True, text=True)
    if check and r.returncode:
        raise RuntimeError(f"gcloud storage {' '.join(args)}\n{r.stderr[-1500:]}")
    return r

PROJECT = sh("gcloud config get-value project 2>/dev/null").stdout.strip()
ACCOUNT = sh("gcloud config get-value account 2>/dev/null").stdout.strip()
probe = WORK / "probe.txt"
probe.write_text(f"write probe {RUN_ID}\n")
WRITE_OK = gcs("cp", str(probe), f"{DATASET}/_runs/{RUN_ID}-probe.txt", check=False).returncode == 0
hb(S, f"project {PROJECT}  account {ACCOUNT}  write to {DATASET}: {'OK' if WRITE_OK else 'FAILED'}")
if STAGE and not WRITE_OK:
    print(f"""
This identity can't write to {DATASET}/. From an account that owns class-demo, run:

  gcloud storage buckets add-iam-policy-binding gs://class-demo \\
    --member="user:{ACCOUNT}" --role=roles/storage.objectAdmin \\
    --condition='expression=resource.name.startsWith("projects/_/buckets/class-demo/objects/alloydb-labs/cymbalfoods-alloydb-workshop/"),title=cymbalfoods-workshop-prefix'

then Run all again. (STAGE = False runs everything except the upload.)""")

---
## 4 · The reference data

The generator starts from hand-curated lists in `datagen/content/`: 163 single-origin spices, herbs, chiles and salts; 143 house blends with their recipes; pantry items (oils, vinegars, sauces, tea, coffee, baking); gift sets; pack sizes; 4 distribution centers and 40 stores; supplier countries; seasonality curves; and a dated calendar of holidays from 2024 through 2027. `content/README.md` says where each list came from.

This cell checks that the lists agree with each other (every blend component exists, every recipe adds up to 100%) and shows the catalog they produce.

In [ ]:
S = "4 · Reference data"
sys.path.insert(0, str(DATAGEN))
from cfgen import content, catalog
c = content.load()
problems = content.check(c)
cat = catalog.build(c)
P = cat["products"]
hb(S, f"content problems: {len(problems)}" + ("" if not problems else "  " + "; ".join(problems[:5])))
hb(S, f"catalog: {len(P):,} products, {len(cat['skus']):,} SKUs, {len(cat['ingredients']):,} ingredients, "
      f"{len(cat['blend_components']):,} blend components")
display(P.groupby("category_id").size().rename("products").to_frame().T)

---
## 5 · Generate

`generate_dataset.py` builds the business in the order it happens:

1. **Catalog and people:** products, pack sizes and prices; stores and staff; 250,000 consumers and 4,000 restaurant accounts, each with a usual basket and contract prices.
2. **Demand and orders:** each product's demand by day (popularity × season × holiday events) drives about 1 million orders and 3 million lines across stores, online and wholesale.
3. **Supply:** the orders are worked backward into production batches, the lots each batch used, and the purchase orders that brought those lots in. So every order line knows which batch, and so which supplier lot, it came from.
4. **Inventory** on Dec 31, 2026 is what the batches have left.
5. **Facts** for the text corpus (reviews, quality reports, account notes, recipes), written later by Gemini in notebook 11.
6. **GA4 clickstream** for 2026, and **validation**.

With `REGENERATE = False` it skips all of that, pulls the staged Parquet back from the bucket, and only re-cuts the slices.

In [ ]:
S = "5 · Generate"
t0 = time.time()
args = [sys.executable, "-u", str(DATAGEN / "generate_dataset.py"), "--out", str(OUT), "--scale", str(SCALE), "--slices", SLICES]
if not RUN_GA4:
    args.append("--no-ga4")
if not REGENERATE:
    if not (OUT / "parquet").exists():
        gcs("rsync", "--recursive", f"{DATASET}/parquet", str(OUT / "parquet"))
        gcs("cp", f"{DATASET}/manifest.json", str(OUT / "manifest.json"))
    if not (OUT / "validation" / "report.json").exists():
        gcs("rsync", "--recursive", f"{DATASET}/validation", str(OUT / "validation"))
    args.append("--cut-only")
proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
tail = []
for line in proc.stdout:
    print(line, end="")
    tail = (tail + [line.rstrip()])[-25:]
rc = proc.wait()
GEN_OK = rc == 0
hb(S, f"exit {rc} after {time.time() - t0:,.0f} s" + ("" if rc != 3 else "  (validation problems; see section 7)"))
if rc not in (0, 3) or not (OUT / "manifest.json").exists():
    # the generator crashed: stop here, with what's needed to diagnose it, rather than fail later on a missing file
    raise RuntimeError(f"generate_dataset.py stopped with exit {rc}. Paste this whole message back.\n"
                       f"python {sys.version.split()[0]}, numpy {numpy.__version__}, pandas {pandas.__version__}, "
                       f"pyarrow {pyarrow.__version__}, duckdb {duckdb.__version__}\n"
                       "last lines:\n" + "\n".join(tail))
MANIFEST = json.loads((OUT / "manifest.json").read_text())
hb(S, "timings (s): " + ", ".join(f"{k} {v}" for k, v in MANIFEST.get("timings", {}).items()))

---
## 6 · A look at the data

DuckDB reads the Parquet files directly, so these queries run with no database. They are the same shape of query the labs run in AlloyDB.

- **Row counts** for every table.
- **One wholesale order:** a restaurant's order with contract prices, and the batch each line shipped from.
- **The holiday rush:** online orders by hour on the busiest day of the three years.
- **A lot trace:** in October 2026 a lot of smoked paprika failed a quality retest. From its lot code, the batches it went into, the products those batches made, and how many customers received them.

In [ ]:
S = "6 · Data"
con = duckdb.connect()
con.execute("SET TimeZone = 'UTC'")
for d in sorted((OUT / "parquet").iterdir()):
    if d.is_dir() and not d.name.startswith("_"):
        con.execute(f"CREATE OR REPLACE VIEW {d.name} AS SELECT * FROM read_parquet('{d}/*.parquet')")
counts = con.execute(" UNION ALL ".join(f"SELECT '{d.name}' AS table_name, count(*) AS rows FROM {d.name}"
                     for d in sorted((OUT / "parquet").iterdir()) if d.is_dir() and not d.name.startswith("_"))).df()
display(counts)
hb(S, "rows: " + ", ".join(f"{r.table_name} {r.rows:,}" for r in counts.itertuples()))

print("One wholesale order, with contract prices and the batch each line came from:")
display(con.execute("""
    SELECT o.order_number, a.business_name, a.city, l.line_no, p.name AS product, s.pack_description,
           l.quantity, l.unit_price, l.price_source, b.batch_code
    FROM orders o JOIN restaurant_accounts a USING (account_id) JOIN order_lines l USING (order_id)
    JOIN skus s USING (sku_id) JOIN products p USING (product_id) LEFT JOIN production_batches b USING (batch_id)
    WHERE o.order_id = (SELECT order_id FROM orders WHERE channel = 'wholesale' AND ordered_at >= TIMESTAMPTZ '2026-10-05'
                        ORDER BY order_number LIMIT 1)
    ORDER BY l.line_no""").df())

print("The busiest online day, by hour (US Eastern):")
peak = con.execute("""
    WITH d AS (SELECT CAST(ordered_at AT TIME ZONE 'America/New_York' AS DATE) AS day, count(*) AS n FROM orders
               WHERE channel = 'online' GROUP BY 1 ORDER BY 2 DESC, 1 LIMIT 1)
    SELECT (SELECT day FROM d) AS day, hour(o.ordered_at AT TIME ZONE 'America/New_York') AS hour, count(*) AS orders
    FROM orders o WHERE o.channel = 'online' AND CAST(o.ordered_at AT TIME ZONE 'America/New_York' AS DATE) = (SELECT day FROM d)
    GROUP BY 1, 2 ORDER BY 2""").df()
display(peak.pivot(index="day", columns="hour", values="orders"))
hb(S, f"peak online day {str(peak.day.iloc[0])[:10]}: {int(peak.orders.sum()):,} orders, busiest hour {int(peak.loc[peak.orders.idxmax(), 'hour'])}:00 ET")

print("From one failed lot to the products and customers it reached:")
trace = con.execute("""
    WITH lot AS (SELECT lot_id, lot_code, ingredient_id, received_at FROM inbound_lots WHERE lot_code = 'L-ES-2608-0117'),
         b AS (SELECT DISTINCT bi.batch_id FROM batch_inputs bi JOIN lot USING (lot_id))
    SELECT p.name AS product, count(DISTINCT pb.batch_id) AS batches, count(DISTINCT l.order_id) AS orders,
           count(DISTINCT o.account_id) AS restaurants, count(DISTINCT o.customer_id) AS consumers
    FROM b JOIN production_batches pb USING (batch_id) JOIN products p USING (product_id)
    LEFT JOIN order_lines l ON l.batch_id = b.batch_id LEFT JOIN orders o USING (order_id)
    GROUP BY 1 ORDER BY orders DESC""").df()
display(trace)

---
## 7 · Validation

The generator checks its own output before anything is staged:

- **Keys:** every primary key is unique.
- **Referential integrity:** every foreign key value has a parent row (no orphans).
- **Consistency:** order lines add up to the order subtotal; totals follow subtotal − discount + shipping + tax; nothing is dated after Dec 31, 2026; no order predates its customer; no batch used a lot before the lot arrived; every blend recipe sums to 100%.
- **Planted moments:** every teaching moment the labs depend on is present, found by the query a lab would use. The answer key is `validation/plants.json`.

In [ ]:
S = "7 · Validation"
REPORT = json.loads((OUT / "validation" / "report.json").read_text())
PLANTS = REPORT["plants"]
hb(S, f"problems: {len(REPORT['problems'])}" + ("" if not REPORT["problems"] else "  " + " | ".join(REPORT["problems"])))
hb(S, "duplicate keys: " + str(sum(REPORT["keys"].values())) + "   foreign-key orphans: " + str(sum(REPORT["foreign_keys"].values())))
hb(S, "checks: " + ", ".join(f"{k}={v}" for k, v in REPORT["checks"].items()))
p8 = PLANTS["P8_failed_lot"]
hb(S, f"P8 failed lot: {p8['restaurants']} restaurants, {p8['consumers']} consumers, {p8['stores']} stores, {p8['orders']} orders, "
      f"{len(p8['products'])} products, {p8['batches_on_hold']} batches on hold")
p13 = PLANTS["P13_stockout"]
hb(S, f"P13 Saigon at DC-CE on hand {p13['saigon_on_hand_dc_ce']} (elsewhere {p13['saigon_on_hand_elsewhere']}), substituted {p13['substituted_lines']}, "
      f"backordered {p13['backordered_lines']}, open PO {p13['open_saigon_po']}; gift last sale {p13['gift_sold_out_last_sale']}, "
      f"views after sell-out {p13.get('ga4_views_after_sellout')}")
if "P14_ga4_gap" in PLANTS:
    hb(S, f"P14 GA4 gap: {PLANTS['P14_ga4_gap']['without_purchase_event']:,} of {PLANTS['P14_ga4_gap']['online_orders_2026']:,} online orders ({PLANTS['P14_ga4_gap']['share']:.1%})")
p15 = PLANTS["P15_peak"]
hb(S, f"P15 online top days {p15['online_top_days'][:3]}; Nov 2026 daily mean {p15['november_2026_online_daily_mean']:.0f}; store top {p15['store_top_days'][:2]}")
hb(S, f"F1 ring addresses (hash, customers, orders, cancelled, cleared): {PLANTS['F1_fraud_ring']['rows']}")
hb(S, f"P6 review themes: {PLANTS.get('P6_review_themes', {}).get('rows')}")
hb(S, f"P7 clumping (humid DC batches?, reviews, %): {PLANTS.get('P7_clumping_by_dc', {}).get('rows')}")
hb(S, f"P11 look-alike names {PLANTS['P11_lookalike_accounts']}")
if "ga4" in REPORT:
    hb(S, f"GA4: {REPORT['ga4']['events']:,} events over {REPORT['ga4']['days']} days")

---
## 8 · Slices

No lab loads everything. Each module loads a **slice**: some groups of tables, optionally a date window, optionally a cap on consumers. Slices are defined in `datagen/slices.yaml`, and each becomes a folder of headerless, gzipped CSV files (the form AlloyDB's import and PostgreSQL's `COPY` both read) plus a `manifest.json` the lab's setup VM follows: which extensions to create, which schema files to run, the load order, each file's rows and checksum, the foreign keys that apply to the tables present, and the index files to build after the load.

A slice is always referentially whole: a kept order line keeps the batch it shipped from, a kept batch keeps its lots, and so on.

To change a slice, edit `slices.yaml`, set `REGENERATE = False`, and run this notebook again.

In [ ]:
S = "8 · Slices"
import pandas as pd
rows = []
for name, m in MANIFEST.get("slices", {}).items():
    rows.append(dict(slice=name, rows=m["rows"], files=m["files"], MB=round(m["bytes"] / 1e6, 1),
                     est_copy_s=m["estimates"]["copy_seconds"], est_import_s=m["estimates"]["import_seconds"]))
    hb(S, f"{name}: {m['rows']:,} rows, {m['files']} files, {m['bytes'] / 1e6:,.1f} MB")
display(pd.DataFrame(rows))
print("Load-time estimates are rough (calibrated on an earlier dataset); the load test (notebook 13) measures them on AlloyDB.")

---
## 9 · Stage to Cloud Storage

Everything goes to `dataset/v1/` in the course bucket. `v1` is pinned by the labs, so a later regeneration goes to a new version and never changes a running lab. Slice folders are synchronized exactly (files a re-cut removed are deleted there); everything else is copied.

In [ ]:
S = "9 · Stage"
if not STAGE:
    hb(S, "skipped (STAGE = False)")
elif not WRITE_OK:
    hb(S, "skipped: no write access (see section 3)")
else:
    t0 = time.time()
    parts = ["sql", "bigquery", "validation"] + (["parquet", "ga4"] if REGENERATE else [])
    for p in parts:
        if (OUT / p).exists():
            gcs("rsync", "--recursive", str(OUT / p), f"{DATASET}/{p}")
    for name in MANIFEST.get("slices", {}):
        gcs("rsync", "--recursive", "--delete-unmatched-destination-objects", str(OUT / "slices" / name), f"{DATASET}/slices/{name}")
    gcs("cp", str(OUT / "manifest.json"), f"{DATASET}/manifest.json")
    du = gcs("du", "-s", "--readable-sizes", f"{DATASET}/").stdout.strip()
    hb(S, f"staged to {DATASET}/ in {time.time() - t0:,.0f} s  ({du.split()[0] if du else '?'} {du.split()[1] if len(du.split()) > 1 else ''})")
    for name in MANIFEST.get("slices", {}):
        ok = gcs("ls", f"{DATASET}/slices/{name}/manifest.json", check=False).returncode == 0
        hb(S, f"  slices/{name}/manifest.json {'present' if ok else 'MISSING'}")

---
## 10 · Handback

One box with everything from this run, and a **Copy** button. If your browser blocks the button, the text is selected for you; copy it with **Edit → Copy**. The same text is saved to the bucket under `dataset/v1/_runs/`.

In [ ]:
from IPython.display import HTML, display
lines = ["=" * 72, f"CYMBAL FOODS DATASET: 10 HANDBACK  run {RUN_ID}", "=" * 72]
for sec, ls_ in HB.items():
    lines += ["", f"## {sec}", *ls_]
writes = [x for x in GCS_LOG if x.split()[2] in ("cp", "rsync", "rm")]
lines += ["", "## Bucket audit", f"{len(GCS_LOG)} calls, {len(writes)} writes, all under {DATASET}/"
          if all(DATASET in w for w in writes) else "WRITE OUTSIDE THE DATASET PREFIX: " + " | ".join(writes)]
text = "\n".join(lines)
(WORK / "handback.txt").write_text(text)
if STAGE and WRITE_OK:
    gcs("cp", str(WORK / "handback.txt"), f"{DATASET}/_runs/{RUN_ID}-handback.txt", check=False)
display(HTML(f"""
<button onclick="navigator.clipboard.writeText(document.getElementById('hb10').innerText).then(()=>this.innerText='Copied')"
  style="padding:6px 14px;margin-bottom:6px;cursor:pointer">Copy</button>
<pre id="hb10" style="white-space:pre-wrap;border:1px solid #888;padding:10px;max-height:520px;overflow:auto">{html.escape(text)}</pre>
<script>var r=document.createRange();r.selectNodeContents(document.getElementById('hb10'));var s=window.getSelection();s.removeAllRanges();s.addRange(r);</script>
"""))